# nb05 — intramonth T-minus path (Session 4, Task 5)

One recent CPI print's **nowcast-as-of-day** path from T-30 to the T-3 freeze, against the
realized first-release value, annotated with each information arrival. This is the operational
answer to real-time tracking: *how early do we know what we know*. Logic lives in
`src/nowcast/intramonth.py` (package-final); this notebook imports and displays.

**Desk-facing — show only after the Checkpoint-2 admission sign-off.** Degraded feature set
(no Keepa goods panel).

In [ ]:
import sys; sys.path.insert(0, '../src')
import datetime as dt, sqlite3
import pandas as pd
from nowcast import intramonth as IM

REF = '2026-06-01'          # the print to trace (a recent, released CPI reference month)
AGG = 'headline'
DB = '../data/db/nowcast.sqlite'
conn = sqlite3.connect(DB)
rel = IM._release_date(conn, REF)
realized = IM._actual_nsa(conn, 'CUUR0000SA0', REF)
conn.close()
print(f'{REF} headline NSA — released {rel}, realized first-release {realized*1e4:+.1f} bp')

In [ ]:
# The T-minus path (estimate at each day) vs the realized value
path = IM.tminus_path(REF, AGG, tmax=30, tmin=3, db_path=DB)
df = pd.DataFrame([{ 'days_to_release': p['days_to_release'], 'as_of': p['as_of'],
                     'estimate_bp': round(p['forecast_mom']*1e4, 1), 'frozen': p['frozen'] }
                   for p in path])
df['realized_bp'] = round(realized*1e4, 1)
df['error_bp'] = (df['estimate_bp'] - df['realized_bp']).round(1)
df

## Information-arrival annotations (from the availability calendar)

- **~T-25** Manheim mid-month update (used-cars early read, days 1-15)
- **rolling through the month** EIA weekly gasoline — the month-M path fills in; the estimate
  converges as the full-month mean stabilizes
- **~T-4** Manheim full-month + NADAC land — the **last useful inputs**
- **T-3 → T-0** FROZEN: nothing new arrives, so the estimate is held at its T-4 value (enforced,
  not just documented)
- **T-0** the official CPI print

In [ ]:
# The honest curve across the last 18 prints: MAE as a function of days-to-release
curve = IM.backtest_curve(AGG, n_prints=18, tmax=30, tmin=3, db_path=DB)
pd.Series(curve['mae_by_days_to_release']).sort_index(ascending=False).rename('MAE bp').to_frame()

**Reading it.** The estimate converges from ~10 bp at T-30 to ~7.5 bp by ~T-8 (as the month's
gasoline path completes and Manheim lands) and improves nothing after T-4 — the freeze is where
the calculator has learned everything it can before the print. ~7.5 bp is the recent-regime
intramonth floor; on the fuller 2019-26 window (with the 2021-22 surge) it is ~11.6 bp. The
prediction layer's honest claim is a **converging ±7-12 bp headline track that freezes at T-4**,
not a point miracle. See `docs/evaluation_1.md` for the full verdicts.